In [0]:
# ===========================================
# Pokemon Lakehouse Setup
# ===========================================

print("Spark Version :", spark.version)

spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").show()

In [0]:
display(
    spark.sql(
        "SHOW CATALOGS"
    )
)

In [0]:
# ===========================================
# Project configuration
# ===========================================

CATALOG_NAME = "pokemon"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

print(f"Catalog : {CATALOG_NAME}")
print(f"Bronze  : {CATALOG_NAME}.{BRONZE_SCHEMA}")
print(f"Silver  : {CATALOG_NAME}.{SILVER_SCHEMA}")
print(f"Gold    : {CATALOG_NAME}.{GOLD_SCHEMA}")

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}
COMMENT 'Raw data collected from Pokemon official websites'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}
COMMENT 'Parsed and normalized tournament and deck data'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{GOLD_SCHEMA}
COMMENT 'Analytics, archetypes, meta share and novelty scores'
""")

print("Schemas created successfully.")

In [0]:
display(
    spark.sql(
        f"SHOW SCHEMAS IN {CATALOG_NAME}"
    )
)

In [0]:
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

display(
    spark.sql("""
    SELECT
        current_catalog() AS catalog,
        current_schema() AS schema
    """)
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{CATALOG_NAME}.{BRONZE_SCHEMA}.event_result_raw
(
    tournament_id STRING NOT NULL,
    source_url STRING NOT NULL,
    http_status INT,
    payload STRING NOT NULL,
    payload_format STRING NOT NULL,
    response_hash STRING NOT NULL,
    scraped_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Raw JSON responses from Pokemon tournament result API'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{CATALOG_NAME}.{BRONZE_SCHEMA}.deck_raw
(
    deck_code STRING NOT NULL,
    source_url STRING NOT NULL,
    http_status INT,
    payload STRING NOT NULL,
    payload_format STRING NOT NULL,
    response_hash STRING NOT NULL,
    scraped_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Raw HTML responses from Pokemon deck print pages'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{CATALOG_NAME}.{BRONZE_SCHEMA}.scrape_log
(
    run_id STRING NOT NULL,
    source_type STRING NOT NULL,
    source_id STRING,
    request_url STRING NOT NULL,
    http_status INT,
    status STRING NOT NULL,
    elapsed_ms BIGINT,
    records_found INT,
    error_type STRING,
    error_message STRING,
    scraped_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Scraping execution logs'
""")

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {CATALOG_NAME}.{BRONZE_SCHEMA}"
    )
)

In [0]:
display(
    spark.sql(f"""
    DESCRIBE DETAIL
    {CATALOG_NAME}.{BRONZE_SCHEMA}.event_result_raw
    """)
)